0. Setup

In [ ]:
!pip -q install -U langchain langchain-community langchain-text-splitters langchain-openai chromadb pypdf

import os
from pathlib import Path

from google.colab import userdata, files

from google.colab import drive
drive.mount("/content/drive")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 10.2 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI")
assert os.environ.get("OPENAI_API_KEY"), "Missing OpenAI secret in Colab."

1. Set up document processing for retrieval, including chunking, embeddings, and a vector database.

set path

In [ ]:
DOCS_DIR = Path("/content/drive/MyDrive/Special Topics LLM Project/Milestone 7 Documents")

assert DOCS_DIR.exists(), f"Docs folder not found: {DOCS_DIR}"
print("Found files:")
for f in DOCS_DIR.iterdir():
    print(" -", f.name)

Found files:
 - kpi_schema_and_definitions.pdf
 - past_rca_scenarios.pdf
 - marketing_spend_impact_framework.pdf
 - mobile_conversion_patterns.pdf
 - anomaly_investigation_runbook.pdf


load files

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
docs = []
for fp in DOCS_DIR.rglob("*.pdf"):
    loader = PyPDFLoader(str(fp))
    loaded_pages = loader.load()  # list[Document], usually one per page
    for d in loaded_pages:
        d.metadata.update({
            "source": fp.name,
            "source_path": str(fp),
            "source_type": "kpi_kb_pdf",
        })
    docs.extend(loaded_pages)

print("Loaded pages:", len(docs))
print("Example metadata:", docs[0].metadata if docs else None)
print("Example preview:", docs[0].page_content[:250] if docs else None)

Loaded pages: 10
Example metadata: {'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'TextEdit', 'creationdate': "D:20260302224643Z00'00'", 'title': 'kpi_schema_and_definitions', 'moddate': "D:20260302224643Z00'00'", 'source': 'kpi_schema_and_definitions.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_path': '/content/drive/MyDrive/Special Topics LLM Project/Milestone 7 Documents/kpi_schema_and_definitions.pdf', 'source_type': 'kpi_kb_pdf'}
Example preview: E-Commerce KPI Deﬁnitions
This document deﬁnes KPIs used in the Oops AI anomaly detection system.
Dataset Columns:
- date
- revenue
- orders
- traffic
- conversion_rate
- avg_order_value
- region
- device_type
- marketing_spend
- product_category
---


chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

splits = splitter.split_documents(docs)

print("Total chunks:", len(splits))
if splits:
    print("Example chunk source:", splits[0].metadata.get("source"),
          "| page:", splits[0].metadata.get("page"),
          "| start_index:", splits[0].metadata.get("start_index"))
    print(splits[0].page_content[:300])

Total chunks: 10
Example chunk source: kpi_schema_and_definitions.pdf | page: 0 | start_index: 0
E-Commerce KPI Deﬁnitions
This document deﬁnes KPIs used in the Oops AI anomaly detection system.
Dataset Columns:
- date
- revenue
- orders
- traffic
- conversion_rate
- avg_order_value
- region
- device_type
- marketing_spend
- product_category
---
## Revenue
Column: revenue
Revenue = orders × avg


In [ ]:
embeddings + vector database

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

PERSIST_DIR = "/content/drive/MyDrive/rag_chroma_database"

vector_db = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)

print("Vector DB size:", vector_db._collection.count())

Vector DB size: 10


check retrieval

In [ ]:
q = "Define conversion_rate and list common drivers of changes in it."
hits = vector_db.similarity_search(q, k=3)

for i, h in enumerate(hits, 1):
    print(f"\n--- Hit {i} ---")
    print("Source:", h.metadata.get("source"), "| page:", h.metadata.get("page"))
    print(h.page_content[:300])


--- Hit 1 ---
Source: kpi_schema_and_definitions.pdf | page: 1
If orders drop while traffic is stable:
→ Likely conversion issue.
If orders and traffic drop together:
→ Likely marketing or acquisition issue.
---
## Traffic
Column: traffic
Traffic represents total sessions.
Traffic changes may result from:
- marketing_spend changes
- campaign pauses
- regional b

--- Hit 2 ---
Source: mobile_conversion_patterns.pdf | page: 0
# Mobile Conversion Rate Failure Patterns
Applies when:
device_type = mobile AND conversion_rate drops signiﬁcantly.
Common causes:
1. Checkout form validation errors
2. Payment gateway integration issues
3. App version release bugs
4. Page load times exceeding 4 seconds
5. Broken promo code logic
6

--- Hit 3 ---
Source: kpi_schema_and_definitions.pdf | page: 3
Common values:
- mobile
- desktop
Device-speciﬁc conversion issues are common root causes.
---
## Product Category
Column: product_category
Revenue shifts may be driven by:
- Category demand changes
- Inven

2. Implement retrieval mechanisms using similarity search, or other appropriate similarity measure.

similarity search

In [ ]:
def retrieve_similarity(query: str, k: int = 5):
    """Dense retrieval via vector similarity search."""
    hits = vector_db.similarity_search(query, k=k)
    return hits

def show_hits(hits, max_preview_chars=350):
    for i, h in enumerate(hits, 1):
        src = h.metadata.get("source")
        page = h.metadata.get("page")
        start = h.metadata.get("start_index")
        print(f"\n--- Hit {i} ---")
        print(f"Source: {src} | page: {page} | start_index: {start}")
        print(h.page_content[:max_preview_chars])

similarity search with score

In [ ]:
hits_scored = vector_db.similarity_search_with_score(anomaly_query, k=5)

print("\nSimilarity search w/ scores (lower distance = more similar in Chroma):")
for i, (doc, score) in enumerate(hits_scored, 1):
    print(f"\n--- Hit {i} --- score: {score}")
    print("Source:", doc.metadata.get("source"), "| page:", doc.metadata.get("page"))
    print(doc.page_content[:300])


Similarity search w/ scores (lower distance = more similar in Chroma):

--- Hit 1 --- score: 0.792878270149231
Source: past_rca_scenarios.pdf | page: 0
# Past RCA Scenario Library (Synthetic Examples)
## Scenario A: Revenue Drop with Stable AOV
Observed:
- revenue ↓ 12%
- avg_order_value stable
- conversion_rate ↓ 10%
Root cause:
Mobile checkout bug.
---
## Scenario B: Region-Speciﬁc Revenue Decline
Observed:
- revenue ↓ 20% in West
- marketing_spe

--- Hit 2 --- score: 0.8220043182373047
Source: kpi_schema_and_definitions.pdf | page: 1
If orders drop while traffic is stable:
→ Likely conversion issue.
If orders and traffic drop together:
→ Likely marketing or acquisition issue.
---
## Traffic
Column: traffic
Traffic represents total sessions.
Traffic changes may result from:
- marketing_spend changes
- campaign pauses
- regional b

--- Hit 3 --- score: 0.8354264497756958
Source: kpi_schema_and_definitions.pdf | page: 2
---
## Average Order Value
Column: avg_order_value
avg_order_valu

query

In [ ]:
anomaly_query = (
    "Revenue dropped 15% starting 2024-04-20. "
    "Mobile conversion_rate dropped 22% (r=0.71, p<0.01). "
    "West marketing_spend decreased 35%. "
    "avg_order_value stable; product_category mix stable."
)

hits_sim = retrieve_similarity(anomaly_query, k=5)
print("Similarity search results:")
show_hits(hits_sim)

Similarity search results:

--- Hit 1 ---
Source: past_rca_scenarios.pdf | page: 0 | start_index: 0
# Past RCA Scenario Library (Synthetic Examples)
## Scenario A: Revenue Drop with Stable AOV
Observed:
- revenue ↓ 12%
- avg_order_value stable
- conversion_rate ↓ 10%
Root cause:
Mobile checkout bug.
---
## Scenario B: Region-Speciﬁc Revenue Decline
Observed:
- revenue ↓ 20% in West
- marketing_spend ↓ 30% in West
- trafﬁc ↓ 25% in West
Root cause

--- Hit 2 ---
Source: kpi_schema_and_definitions.pdf | page: 1 | start_index: 0
If orders drop while traffic is stable:
→ Likely conversion issue.
If orders and traffic drop together:
→ Likely marketing or acquisition issue.
---
## Traffic
Column: traffic
Traffic represents total sessions.
Traffic changes may result from:
- marketing_spend changes
- campaign pauses
- regional budget shifts
- seasonality
- technical outages
Tra

--- Hit 3 ---
Source: kpi_schema_and_definitions.pdf | page: 2 | start_index: 0
---
## Average Order Value
Column: a

3. Integrate the retrieved context into the generation process to improve response quality.

OpenAI

In [ ]:
from langchain_openai import ChatOpenAI

MODEL = "gpt-4.1-nano"  # your chosen generation model
llm = ChatOpenAI(model=MODEL)

Format retrieved docs into a block with citations

In [ ]:
def format_context(docs, max_chars=6000):
    blocks = []
    total = 0
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source")
        page = d.metadata.get("page")
        block = f"[{i}] Source: {src} (page {page})\n{d.page_content}".strip()
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block)
    return "\n\n".join(blocks)

input

In [ ]:
anomaly_summary = (
    "Dataset columns: date, revenue, orders, traffic, conversion_rate, avg_order_value, region, device_type, "
    "marketing_spend, product_category.\n\n"
    "Anomaly: Revenue dropped 15% starting 2024-04-20.\n"
    "Findings: Primary driver: Mobile conversion_rate dropped 22% (r=0.71, p<0.01). "
    "Secondary: marketing_spend in the West region decreased 35%. "
    "Ruled out: avg_order_value (no significant change), product_category mix (stable)."
)

retrieve context

In [ ]:
retrieved = vector_db.similarity_search(anomaly_summary, k=5)
context = format_context(retrieved, max_chars=6000)

RAG prompt

In [ ]:
rag_prompt = f"""You are an analytics assistant generating executive-facing explanations.

Rules:
- Use ONLY the evidence in (1) the anomaly summary and (2) the retrieved context.
- Clearly separate evidence from hypotheses.
- Do not introduce causes not supported by the anomaly summary or retrieved context.
- If the knowledge base does not support a claim, say: "Not supported by retrieved context."
- When referencing retrieved info, cite the bracketed source number(s) like [1], [2].

Retrieved context:
{context}

Anomaly summary:
{anomaly_summary}

Output format:
Executive summary (2-3 sentences)
Key evidence
Plausible explanations (clearly labeled as hypotheses)
Suggested next analysis
"""

rag_response = llm.invoke(rag_prompt).content
print("=== WITH RAG ===\n")
print(rag_response)

=== WITH RAG ===

Executive Summary:
Between April 20, 2024, and the present, revenue declined by 15%, primarily driven by a 22% drop in mobile conversion rate. The decrease in marketing spend in the West region likely contributed to the revenue downturn, while other factors such as average order value and product category mix remained stable.

Key Evidence:
- Mobile conversion rate dropped by 22% (r=0.71, p<0.01) [Anomaly summary]
- Marketing spend in the West region decreased by 35% [Anomaly summary]
- Average order value showed no significant change [Anomaly summary]
- Product category mix remained stable [Anomaly summary]

Plausible Explanations:
- The decline in mobile conversion rate suggests device-specific UX issues or checkout problems for mobile users (supported by the importance of device_type and conversion_rate changes) [2]
- The reduction in marketing spend in the West region may have caused decreased traffic or brand visibility in that region, indirectly impacting overal

baseline prompt

In [ ]:
baseline_prompt = f"""You are an analytics assistant generating executive-facing explanations.

Rules:
- Base claims strictly on the anomaly summary only.
- Clearly separate evidence from hypotheses.
- Do not introduce causes not supported by the anomaly summary.
- If unsure, say "I don't know."

Anomaly summary:
{anomaly_summary}

Output format:
Executive summary (2-3 sentences)
Key evidence
Plausible explanations (clearly labeled as hypotheses)
Suggested next analysis
"""

baseline_response = llm.invoke(baseline_prompt).content
print("=== BASELINE ===\n")
print(baseline_response)

=== BASELINE ===

Executive summary:
Starting April 20, 2024, there was a 15% decrease in revenue. The primary factor appears to be a significant decline in mobile conversion rates, accompanied by reduced marketing spend in the West region.

Key evidence:
- Mobile conversion rate declined by 22% starting on 2024-04-20, with a strong correlation (r=0.71, p<0.01).
- Marketing spend in the West region decreased by 35% around the same period.
- No significant change observed in average order value.
- The product category mix remained stable.

Plausible explanations:
- The drop in mobile conversion rate likely contributed substantially to the revenue decline.
- Reduced marketing spend in the West region may have decreased overall traffic and customer acquisition in that area, affecting revenue.

Suggested next analysis:
- Examine traffic trends and traffic sources, especially in the West region and on mobile devices.
- Analyze whether the decrease in marketing spend directly impacted traffi

comparison

In [ ]:
print("\n=== COMPARISON ===")
print("RAG response length:", len(rag_response))
print("Baseline response length:", len(baseline_response))
print("Retrieved sources used:", [d.metadata.get("source") for d in retrieved])


=== COMPARISON ===
RAG response length: 1417
Baseline response length: 1149
Retrieved sources used: ['kpi_schema_and_definitions.pdf', 'kpi_schema_and_definitions.pdf', 'past_rca_scenarios.pdf', 'kpi_schema_and_definitions.pdf', 'kpi_schema_and_definitions.pdf']


4. Evaluate the retrieval quality using appropriate metrics (e.g., precision@k, ecall) and assess answer quality.

chunk IDs for labeling

In [ ]:
def chunk_id(d):
    return f"{d.metadata.get('source')}::page={d.metadata.get('page')}::start={d.metadata.get('start_index')}"

In [ ]:
def inspect_retrieval(query, k=8):
    hits = vector_db.similarity_search(query, k=k)
    print("QUERY:", query)
    for i, h in enumerate(hits, 1):
        print(f"\n--- {i} ---")
        print("chunk_id:", chunk_id(h))
        print("source:", h.metadata.get("source"), "| page:", h.metadata.get("page"))
        print(h.page_content[:350])
    return hits

In [ ]:
inspect_retrieval("What is conversion_rate and what drives changes in it?", k=8)

QUERY: What is conversion_rate and what drives changes in it?

--- 1 ---
chunk_id: kpi_schema_and_definitions.pdf::page=1::start=0
source: kpi_schema_and_definitions.pdf | page: 1
If orders drop while traffic is stable:
→ Likely conversion issue.
If orders and traffic drop together:
→ Likely marketing or acquisition issue.
---
## Traffic
Column: traffic
Traffic represents total sessions.
Traffic changes may result from:
- marketing_spend changes
- campaign pauses
- regional budget shifts
- seasonality
- technical outages
Tra

--- 2 ---
chunk_id: mobile_conversion_patterns.pdf::page=0::start=0
source: mobile_conversion_patterns.pdf | page: 0
# Mobile Conversion Rate Failure Patterns
Applies when:
device_type = mobile AND conversion_rate drops signiﬁcantly.
Common causes:
1. Checkout form validation errors
2. Payment gateway integration issues
3. App version release bugs
4. Page load times exceeding 4 seconds
5. Broken promo code logic
6. Cart timeout errors
7. Mobile-speciﬁc inventory 


[Document(metadata={'source_path': '/content/drive/MyDrive/Special Topics LLM Project/Milestone 7 Documents/kpi_schema_and_definitions.pdf', 'page_label': '2', 'source': 'kpi_schema_and_definitions.pdf', 'page': 1, 'start_index': 0, 'creator': 'TextEdit', 'moddate': "D:20260302224643Z00'00'", 'source_type': 'kpi_kb_pdf', 'total_pages': 4, 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'title': 'kpi_schema_and_definitions', 'creationdate': "D:20260302224643Z00'00'"}, page_content='If orders drop while traffic is stable:\n→ Likely conversion issue.\nIf orders and traffic drop together:\n→ Likely marketing or acquisition issue.\n---\n## Traffic\nColumn: traffic\nTraffic represents total sessions.\nTraffic changes may result from:\n- marketing_spend changes\n- campaign pauses\n- regional budget shifts\n- seasonality\n- technical outages\nTraffic declines isolated to one region:\n→ Investigate regional marketing_spend.\n---\n## Conversion Rate\nColumn: conversion_rate\nco

In [ ]:
inspect_retrieval("How does marketing_spend affect revenue by region?", k=8)

QUERY: How does marketing_spend affect revenue by region?

--- 1 ---
chunk_id: marketing_spend_impact_framework.pdf::page=0::start=0
source: marketing_spend_impact_framework.pdf | page: 0
# Marketing Spend Impact Framework
Column: marketing_spend
Marketing spend primarily drives trafﬁc acquisition.
Expected relationships:
If marketing_spend decreases:
→ trafﬁc should decline
→ revenue should decline proportionally
→ conversion_rate often remains stable
If marketing_spend drops only in one region:
→ Revenue drop likely region-speciﬁc

--- 2 ---
chunk_id: kpi_schema_and_definitions.pdf::page=2::start=0
source: kpi_schema_and_definitions.pdf | page: 2
---
## Average Order Value
Column: avg_order_value
avg_order_value = revenue / orders
Drivers:
- Pricing changes
- Discounting strategy
- Product_category mix shifts
- Bundling changes
If avg_order_value is stable:
→ Revenue decline likely not pricing-driven.
---
## Marketing Spend
Column: marketing_spend
marketing_spend impacts traffic acqu

[Document(metadata={'page_label': '1', 'source': 'marketing_spend_impact_framework.pdf', 'source_path': '/content/drive/MyDrive/Special Topics LLM Project/Milestone 7 Documents/marketing_spend_impact_framework.pdf', 'creator': 'TextEdit', 'start_index': 0, 'source_type': 'kpi_kb_pdf', 'page': 0, 'creationdate': "D:20260302224658Z00'00'", 'title': 'marketing_spend_impact_framework', 'total_pages': 1, 'moddate': "D:20260302224658Z00'00'", 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext'}, page_content='# Marketing Spend Impact Framework\nColumn: marketing_spend\nMarketing spend primarily drives trafﬁc acquisition.\nExpected relationships:\nIf marketing_spend decreases:\n→ trafﬁc should decline\n→ revenue should decline proportionally\n→ conversion_rate often remains stable\nIf marketing_spend drops only in one region:\n→ Revenue drop likely region-speciﬁc.\nIf marketing_spend stable but trafﬁc drops:\n→ Investigate channel performance or attribution issues.\nIf trafﬁc sta

In [ ]:
inspect_retrieval("If avg_order_value is stable but revenue drops, what should we check?", k=8)

QUERY: If avg_order_value is stable but revenue drops, what should we check?

--- 1 ---
chunk_id: kpi_schema_and_definitions.pdf::page=2::start=0
source: kpi_schema_and_definitions.pdf | page: 2
---
## Average Order Value
Column: avg_order_value
avg_order_value = revenue / orders
Drivers:
- Pricing changes
- Discounting strategy
- Product_category mix shifts
- Bundling changes
If avg_order_value is stable:
→ Revenue decline likely not pricing-driven.
---
## Marketing Spend
Column: marketing_spend
marketing_spend impacts traffic acquisition

--- 2 ---
chunk_id: kpi_schema_and_definitions.pdf::page=1::start=0
source: kpi_schema_and_definitions.pdf | page: 1
If orders drop while traffic is stable:
→ Likely conversion issue.
If orders and traffic drop together:
→ Likely marketing or acquisition issue.
---
## Traffic
Column: traffic
Traffic represents total sessions.
Traffic changes may result from:
- marketing_spend changes
- campaign pauses
- regional budget shifts
- seasonality
- technic

[Document(metadata={'start_index': 0, 'moddate': "D:20260302224643Z00'00'", 'creator': 'TextEdit', 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'title': 'kpi_schema_and_definitions', 'total_pages': 4, 'creationdate': "D:20260302224643Z00'00'", 'source_type': 'kpi_kb_pdf', 'page': 2, 'source_path': '/content/drive/MyDrive/Special Topics LLM Project/Milestone 7 Documents/kpi_schema_and_definitions.pdf', 'source': 'kpi_schema_and_definitions.pdf', 'page_label': '3'}, page_content='---\n## Average Order Value\nColumn: avg_order_value\navg_order_value = revenue / orders\nDrivers:\n- Pricing changes\n- Discounting strategy\n- Product_category mix shifts\n- Bundling changes\nIf avg_order_value is stable:\n→ Revenue decline likely not pricing-driven.\n---\n## Marketing Spend\nColumn: marketing_spend\nmarketing_spend impacts traffic acquisition.\nIf marketing_spend drops:\n→ Expect traffic decline within 1–3 days.\n→ Revenue impact typically proportional.\nIf revenue drops 

In [ ]:
labled test set

In [ ]:
test_set = [
    {
        "query": "What is conversion_rate and what drives changes in it?",
        "relevant_chunk_ids": [
        "kpi_schema_and_definitions.pdf::page=1::start=0",
        "mobile_conversion_patterns.pdf::page=0::start=0",
        "past_rca_scenarios.pdf::page=0::start=0"
    ]
    },
    {
        "query": "How can marketing_spend changes (especially regional decreases) affect revenue?",
        "relevant_chunk_ids": [
        "marketing_spend_impact_framework.pdf::page=0::start=0",
        "kpi_schema_and_definitions.pdf::page=2::start=0",
        "past_rca_scenarios.pdf::page=0::start=0",
        "kpi_schema_and_definitions.pdf::page=1::start=0"
    ]
    },
    {
        "query": "If avg_order_value is stable but revenue drops, what should we check?",
        "relevant_chunk_ids": [
      "kpi_schema_and_definitions.pdf::page=2::start=0",
      "kpi_schema_and_definitions.pdf::page=0::start=0",
      "anomaly_investigation_runbook.pdf::page=0::start=0",
      "past_rca_scenarios.pdf::page=0::start=0",
      "kpi_schema_and_definitions.pdf::page=1::start=0"  # optional, include if you want broader 'what to check'
    ]
    },
]

precision@k

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    retrieved_k = retrieved_ids[:k]
    tp = len(set(retrieved_k) & set(relevant_ids))
    return tp / k

recall@k

In [ ]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    retrieved_k = retrieved_ids[:k]
    tp = len(set(retrieved_k) & set(relevant_ids))
    return tp / len(relevant_ids) if relevant_ids else 0.0

evaluate

In [ ]:
def evaluate_retrieval(test_set, k=5, use_mmr=False):
    scores = []
    for case in test_set:
        rel = case["relevant_chunk_ids"]
        if not rel:
            continue  # skip unlabeled

        if use_mmr:
            hits = vector_db.max_marginal_relevance_search(case["query"], k=k, fetch_k=20)
        else:
            hits = vector_db.similarity_search(case["query"], k=k)

        retrieved_ids = [chunk_id(h) for h in hits]

        p = precision_at_k(retrieved_ids, rel, k)
        r = recall_at_k(retrieved_ids, rel, k)

        scores.append({"query": case["query"], "precision": p, "recall": r})

    if not scores:
        return None, None, []

    avg_p = sum(s["precision"] for s in scores) / len(scores)
    avg_r = sum(s["recall"] for s in scores) / len(scores)
    return avg_p, avg_r, scores

k = 5
avg_p_sim, avg_r_sim, per_q_sim = evaluate_retrieval(test_set, k=k, use_mmr=False)
avg_p_mmr, avg_r_mmr, per_q_mmr = evaluate_retrieval(test_set, k=k, use_mmr=True)

print(f"Similarity: Precision@{k}={avg_p_sim}  Recall@{k}={avg_r_sim}")
print(f"MMR:        Precision@{k}={avg_p_mmr}  Recall@{k}={avg_r_mmr}")

print("\nPer-query (Similarity):")
for row in per_q_sim:
    print("-", row["precision"], row["recall"], "|", row["query"])

print("\nPer-query (MMR):")
for row in per_q_mmr:
    print("-", row["precision"], row["recall"], "|", row["query"])


Similarity: Precision@5=0.7333333333333334  Recall@5=0.9333333333333332
MMR:        Precision@5=0.5333333333333333  Recall@5=0.6722222222222222

Per-query (Similarity):
- 0.6 1.0 | What is conversion_rate and what drives changes in it?
- 0.8 1.0 | How can marketing_spend changes (especially regional decreases) affect revenue?
- 0.8 0.8 | If avg_order_value is stable but revenue drops, what should we check?

Per-query (MMR):
- 0.4 0.6666666666666666 | What is conversion_rate and what drives changes in it?
- 0.6 0.75 | How can marketing_spend changes (especially regional decreases) affect revenue?
- 0.6 0.6 | If avg_order_value is stable but revenue drops, what should we check?


quality assessment

In [ ]:
rubric = {
    "faithfulness": "Are claims supported by anomaly summary + retrieved context? (no hallucinated causes)",
    "relevance": "Does it directly answer the anomaly question and highlight the true drivers?",
    "clarity_exec": "Is it concise, executive-friendly, well-structured?",
    "actionability": "Are next analyses specific and feasible?"
}

In [ ]:
def generate_baseline_and_rag(anomaly_summary, k=5):
    # Retrieve context
    retrieved = vector_db.similarity_search(anomaly_summary, k=k)
    ctx = "\n\n".join(
        f"[{i+1}] Source: {d.metadata.get('source')} (page {d.metadata.get('page')})\n{d.page_content}"
        for i, d in enumerate(retrieved)
    )

    rag_prompt = f"""You are an analytics assistant generating executive-facing explanations.

Rules:
- Use ONLY the evidence in (1) anomaly summary and (2) retrieved context.
- Separate evidence vs hypotheses.
- If unsupported, say "Not supported by retrieved context."
- Cite sources like [1], [2] when using retrieved context.

Retrieved context:
{ctx}

Anomaly summary:
{anomaly_summary}

Output format:
Executive summary (2-3 sentences)
Key evidence
Plausible explanations (clearly labeled as hypotheses)
Suggested next analysis
"""
    baseline_prompt = f"""You are an analytics assistant generating executive-facing explanations.

Rules:
- Base claims strictly on anomaly summary only.
- Separate evidence vs hypotheses.
- If unsure, say "I don't know."

Anomaly summary:
{anomaly_summary}

Output format:
Executive summary (2-3 sentences)
Key evidence
Plausible explanations (clearly labeled as hypotheses)
Suggested next analysis
"""

    rag = llm.invoke(rag_prompt).content
    base = llm.invoke(baseline_prompt).content
    return base, rag, retrieved

anomaly summary

In [ ]:
anomaly_summary = (
    "Anomaly: Revenue dropped 15% starting 2024-04-20.\n"
    "Findings: Primary driver: Mobile conversion_rate dropped 22% (r=0.71, p<0.01). "
    "Secondary: marketing_spend in West decreased 35%. "
    "Ruled out: avg_order_value stable; product_category mix stable."
)

baseline_ans, rag_ans, retrieved_docs = generate_baseline_and_rag(anomaly_summary, k=5)

print("\n====================")
print("WITHOUT RAG")
print("====================\n")
print(baseline_ans)

print("\n====================")
print("WITH RAG")
print("====================\n")
print(rag_ans)

print("\nRetrieved sources used:")
for i, d in enumerate(retrieved_docs, 1):
    print(f"[{i}] {d.metadata.get('source')} (page {d.metadata.get('page')})")

print("\nRubric to score manually (1–5 each):")
for k, v in rubric.items():
    print("-", k + ":", v)


WITHOUT RAG

Executive summary:
Starting around April 20, 2024, revenue declined by 15%. The primary factor appears to be a significant drop in mobile conversion rates, alongside decreased marketing spend in the Western region.

Key evidence:
- Mobile conversion rate decreased by 22% with a strong correlation (r=0.71, p<0.01).
- Marketing spend in the West decreased by 35%.
- Average order value and product category mix remained stable during this period.

Plausible explanations:
- A reduction in marketing efforts in the West could have lowered traffic quality or volume, impacting conversion.
- External factors, such as changes in user behavior on mobile devices, may have contributed to the decline in conversion rates.

Suggested next analysis:
Examine traffic and engagement metrics specifically from mobile devices and the Western region to identify potential causes of the conversion drop. Additionally, review any external factors or events around April 20, 2024, that could influence 

simple check

In [ ]:
def contains_disallowed_claims(text):
    red_flags = ["definitely", "certainly", "proved", "confirmed root cause"]
    return any(rf in text.lower() for rf in red_flags)

print("\nBaseline red-flag language:", contains_disallowed_claims(baseline_ans))
print("RAG red-flag language:", contains_disallowed_claims(rag_ans))


Baseline red-flag language: False
RAG red-flag language: False
